<a href="https://colab.research.google.com/github/sap-tarshi-ghosh/python_/blob/main/03_hyperparameter_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

In [6]:
data = load_breast_cancer()

X = data.data
y = data.target

Define K-Fold

In [7]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

Define Models and Hyperparameters

In [8]:
models = {
    "Decision Tree": (
        DecisionTreeClassifier(random_state=42),
        {
            "max_depth": [None, 5, 10, 20],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4]
        }
    ),

    "KNN": (
        KNeighborsClassifier(),
        {
            "n_neighbors": [3, 5, 7, 9],
            "weights": ["uniform", "distance"],
            "p": [1, 2]   # 1=Manhattan, 2=Euclidean
        }
    ),

    "SVM": (
        SVC(random_state=42),
        {
            "C": [0.1, 1, 10, 100],
            "kernel": ["linear", "rbf"],
            "gamma": ["scale", "auto"]
        }
    )
}


Training Loop

In [9]:
results = {}

for name, (model, params) in models.items():

    print("\n==============================")
    print(f"Model: {name}")
    print("==============================")

    fold_metrics = []

    for train_index, test_index in kf.split(X):

        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]

        # Randomized Search
        random_search = RandomizedSearchCV(
            estimator=model,
            param_distributions=params,
            n_iter=10,
            cv=3,
            scoring="accuracy",
            random_state=42,
            n_jobs=-1
        )

        random_search.fit(X_train, y_train)

        best_model = random_search.best_estimator_

        y_pred = best_model.predict(X_test)

        # Calculate metrics
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='weighted')
        rec = recall_score(y_test, y_pred, average='weighted')
        f1 = f1_score(y_test, y_pred, average='weighted')

        fold_metrics.append({
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1_score": f1
        })

        print("Fold Accuracy:", acc)
        print("Fold Precision:", prec)
        print("Fold Recall:", rec)
        print("Fold F1-Score:", f1)
        print("Best Params in this fold:", random_search.best_params_)

    # Calculate average metrics across folds
    avg_acc = np.mean([m['accuracy'] for m in fold_metrics])
    avg_prec = np.mean([m['precision'] for m in fold_metrics])
    avg_rec = np.mean([m['recall'] for m in fold_metrics])
    avg_f1 = np.mean([m['f1_score'] for m in fold_metrics])

    results[name] = {
        "Accuracy": avg_acc,
        "Precision": avg_prec,
        "Recall": avg_rec,
        "F1-Score": avg_f1
    }

    print(f"\nAverage Accuracy for {name}: {avg_acc:.4f}")
    print(f"Average Precision for {name}: {avg_prec:.4f}")
    print(f"Average Recall for {name}: {avg_rec:.4f}")
    print(f"Average F1-Score for {name}: {avg_f1:.4f}")





Model: Decision Tree
Fold Accuracy: 0.9473684210526315
Fold Precision: 0.9473684210526315
Fold Recall: 0.9473684210526315
Fold F1-Score: 0.9473684210526315
Best Params in this fold: {'min_samples_split': 5, 'min_samples_leaf': 2, 'max_depth': 5}
Fold Accuracy: 0.9649122807017544
Fold Precision: 0.9649122807017544
Fold Recall: 0.9649122807017544
Fold F1-Score: 0.9649122807017544
Best Params in this fold: {'min_samples_split': 2, 'min_samples_leaf': 2, 'max_depth': 5}
Fold Accuracy: 0.9385964912280702
Fold Precision: 0.938457254246728
Fold Recall: 0.9385964912280702
Fold F1-Score: 0.9384499917007656
Best Params in this fold: {'min_samples_split': 10, 'min_samples_leaf': 4, 'max_depth': 20}
Fold Accuracy: 0.9385964912280702
Fold Precision: 0.9390179995443153
Fold Recall: 0.9385964912280702
Fold F1-Score: 0.938731642017737
Best Params in this fold: {'min_samples_split': 5, 'min_samples_leaf': 2, 'max_depth': 5}
Fold Accuracy: 0.9469026548672567
Fold Precision: 0.9469026548672567
Fold Reca

In [10]:
print("Final Model Comparison")


for model_name, metrics in results.items():
    print(f"\nModel: {model_name}")
    for metric_name, value in metrics.items():
        print(f"  {metric_name}: {value:.4f}")

Final Model Comparison

Model: Decision Tree
  Accuracy: 0.9473
  Precision: 0.9473
  Recall: 0.9473
  F1-Score: 0.9473

Model: KNN
  Accuracy: 0.9402
  Precision: 0.9407
  Recall: 0.9402
  F1-Score: 0.9397

Model: SVM
  Accuracy: 0.9578
  Precision: 0.9593
  Recall: 0.9578
  F1-Score: 0.9575
